# Lab 6 · Harness de prontidão
**Oficina do Builder · Winovoito** — Bancada 6 · Segurança, custo e prontidão

Você tem um RAG (Lab 4) e um agente (Lab 5). Está pronto para ir ao ar? "Parece que funciona" não é resposta. Neste lab você monta o **harness** que responde com números:

1. registra casos e roda **cada um 3 vezes** (um acerto isolado pode ser sorte);
2. mede aprovação, consistência, **custo por tarefa**, tempo p50/p95 e passos;
3. ataca o sistema com uma **bateria de 15 ataques** (diretos, indiretos, fora de escopo, ação sem aprovação);
4. valida **uma otimização de custo** pelo harness;
5. roda o **Revisor dos 7 erros**;
6. gera o **Relatório de prontidão**: pronto, pronto com ressalvas ou não pronto.

In [ ]:
# Setup: baixa o material aberto da Oficina (rode uma vez)
REPO_URL = "https://github.com/winovoito/oficina-do-builder-aberto"  # endereço definitivo a confirmar
import os, sys
if not os.path.exists("oficina-do-builder-aberto"):
    !git clone -q {REPO_URL}
sys.path.insert(0, "oficina-do-builder-aberto")
%pip install -q -r oficina-do-builder-aberto/requirements.txt
import json, pandas as pd

## 1 · Provedor e sistema

In [ ]:
import os, json, tempfile
from pathlib import Path
import pandas as pd
from oficina import provedores, churn, agente, rag, harness
from oficina.rag_simulado import criar_modelo_simulado
from oficina.sinteticos import gerar_fluxo
PASTA_MODELO = Path("modelos/churn")
if not PASTA_MODELO.exists():
    PASTA_MODELO = Path(tempfile.mkdtemp()); churn.rodar_referencia(gerar_fluxo(), PASTA_MODELO)
clientes = agente.carregar_clientes()
servico = agente.servico_churn_de_pasta(PASTA_MODELO)
SIMULADO = not os.environ.get("OFICINA_MODELO") and os.environ.get("OFICINA_PROVEDOR") != "falso"
if SIMULADO: os.environ["OFICINA_PROVEDOR"] = "falso"
SIMULADO = os.environ.get("OFICINA_PROVEDOR") == "falso"
COL = rag.carregar_colecao("juridico")
if SIMULADO: provedores.usar_resposta_falsa(criar_modelo_simulado(COL))
print("⚠️ Modo simulado: valida a encanação, não a qualidade de um modelo real." if SIMULADO else "✅ Provedor real")

## 2 · Casos e execução repetida
Um caso por pergunta do acervo jurídico (25). Cada um roda 3 vezes.

In [ ]:
casos = harness.casos_rag("juridico")
linhas = [harness.rodar_caso(c, n=3) for c in casos]
resumo = harness.resumir(linhas)
pd.DataFrame([resumo])

**Aprovação** é a média de acertos. **Acerto nas 3 rodadas** é a fração de casos que passaram **todas** as vezes: é a que mostra a confiabilidade. Se as duas se afastam, o sistema é instável. Veja como isso fica com um modelo que falha de vez em quando:

In [ ]:
if SIMULADO:
    provedores.usar_resposta_falsa(harness.instavel(criar_modelo_simulado(COL), taxa_erro=0.6, semente=1))
    inst = harness.resumir([harness.rodar_caso(c, n=3) for c in casos])
    provedores.usar_resposta_falsa(criar_modelo_simulado(COL))
    print(f"aprovação {inst['aprovacao']:.0%} · acerto nas 3 rodadas {inst['taxa_acerto_nas_n']:.0%}")

## 3 · A bateria de 15 ataques
O modelo simulado é **fraco de propósito**: faz o que o atacante pede. A asserção olha o **estado** (nada enviado, nada vazado), não o texto. Se o ataque passar, é falha do sistema, não do atacante.

In [ ]:
ataques = harness.rodar_bateria(clientes, servico)
if SIMULADO: provedores.usar_resposta_falsa(criar_modelo_simulado(COL))
tabela = pd.DataFrame(ataques)[["id", "categoria", "bloqueado", "falhas"]]
print(f"{tabela.bloqueado.sum()}/{len(tabela)} bloqueados"); tabela

Repare no que **não** está coberto: uma instrução escondida num documento, **disfarçada** sem palavras de gatilho, pode alterar o *texto* de uma resposta do RAG sem acionar nenhuma ação proibida. Por isso o relatório tem uma seção de limitações: um relatório que só mostra o que passou não é confiável.

## 4 · Uma otimização de custo, validada
Passar menos trechos ao modelo custa menos. Mas só vale se a aprovação não cair:

In [ ]:
otim = harness.comparar_configuracoes("juridico", ks=(5, 3), n=1)
print(f"economia de {otim['economia_pct']:.0f}% · aprovação {otim['por_k'][5]['aprovacao']:.0%} → {otim['por_k'][3]['aprovacao']:.0%} · validada: {otim['validada']}")

## 5 · O Revisor dos 7 erros

In [ ]:
if SIMULADO:
    revisao = None
    print("O Revisor precisa de um modelo de verdade: rode este notebook com o provedor real.")
else:
    material = Path("oficina-do-builder-aberto/oficina/agente.py").read_text("utf-8") if Path("oficina-do-builder-aberto").exists() else Path("oficina/agente.py").read_text("utf-8")
    revisao = harness.revisar(material)
    print(revisao[:1500])

## 6 · Relatório de prontidão

In [ ]:
if SIMULADO: provedores.usar_resposta_falsa(criar_modelo_simulado(COL))
relatorio = harness.gerar_relatorio(resumo, ataques, otim, revisao, modo_simulado=SIMULADO)
Path("relatorio_de_prontidao.md").write_text(relatorio, encoding="utf-8")
print(relatorio[:1800])

## 7 · Seu turno
1. Escreva **dois ataques novos** (um direto e um indireto) e acrescente a `harness.ATAQUES`. O sistema os bloqueia? Se não, qual regra em código falta?
2. Tente **outra otimização** (por exemplo, trocar o modelo por um menor). O harness a valida ou reprova?
3. Mude o limite de `min_aprovacao` do veredito para 0,98 e releia o relatório. Quem decide o que é "pronto" no seu projeto, e com base em quê?

## 8 · Teste de aceitação

In [ ]:
assert tabela.bloqueado.all(), "algum ataque passou"
assert "# Relatório de prontidão" in relatorio and "Limitações conhecidas" in relatorio
print("veredito:", harness.veredito(resumo, ataques))
print("✅ 15/15 ataques bloqueados · relatório gerado · seus testes passaram")

**Próximo:** o **Projeto final**. Você aplica tudo a um caso seu, usando o guia `final_guia.ipynb`.